In [48]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_fscore_support
)

from torchvision import transforms
from datasets import load_dataset, load_from_disk
import timm


In [49]:
class CustomAugmenter(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.augment = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor()
        ])

    def forward(self, x):
        return self.augment(x)


In [50]:
class MultiModalNet(nn.Module):
    def __init__(
        self, num_classes, ts_input_dim=5, ts_model='transformer', has_vision=True, has_timeseries=True
    ):
        super().__init__()
        self.has_vision = has_vision
        self.has_timeseries = has_timeseries
        self.ts_model = ts_model.lower()
        self.ts_input_dim = ts_input_dim

        # Vision encoder
        self.vision_output_dim = 1024 if has_vision else 0  # EfficientNet-B0 output is 1280
        if has_vision:
            self.vision_model = timm.create_model(
                "mobilenetv3_small_100.lamb_in1k", pretrained=True, num_classes=0
            )

        # -------- Timeseries Module --------
        self.ts_hidden_dim = 0
        if has_timeseries:
            if self.ts_model == 'gru':
                self.ts_hidden_dim = 256
                self.ts_module = nn.GRU(
                    input_size=ts_input_dim,
                    hidden_size=self.ts_hidden_dim,
                    batch_first=True
                )

            elif self.ts_model == 'lstm':
                self.ts_hidden_dim = 256
                self.ts_module = nn.LSTM(
                    input_size=ts_input_dim,
                    hidden_size=self.ts_hidden_dim,
                    batch_first=True
                )
            
            elif self.ts_model == 'rnn':
                self.ts_hidden_dim = 256
                self.ts_module = nn.RNN(
                    input_size=ts_input_dim,
                    hidden_size=self.ts_hidden_dim,
                    batch_first=True
                )

            elif self.ts_model == 'cnn':
                self.ts_hidden_dim = 256
                self.ts_module = nn.Sequential(
                    nn.Conv1d(ts_input_dim, 256, kernel_size=3, padding=1),
                    nn.ReLU(),
                    nn.Conv1d(256, self.ts_hidden_dim, kernel_size=3, padding=1),
                    nn.ReLU(),
                    nn.AdaptiveAvgPool1d(1),
                )

            elif self.ts_model == 'transformer':
                self.ts_embedding_dim = 128
                self.ts_proj = nn.Linear(ts_input_dim, self.ts_embedding_dim)

                encoder_layer = nn.TransformerEncoderLayer(
                    d_model=self.ts_embedding_dim,
                    nhead=4,
                    dim_feedforward=128,
                    dropout=0.1,
                    batch_first=True,
                    norm_first=True
                )
                self.ts_module = nn.TransformerEncoder(encoder_layer, num_layers=4)

                # Attention-based pooling
                self.ts_pool = nn.Sequential(
                    nn.Linear(self.ts_embedding_dim, 64),
                    nn.Tanh(),
                    nn.Linear(64, 1)
                )
                self.ts_hidden_dim = self.ts_embedding_dim

            else:
                raise ValueError(f"Unsupported ts_model: {ts_model}")

        # -------- Classifier --------
        self.combined_dim = self.vision_output_dim + self.ts_hidden_dim
        self.fc = nn.Sequential(
            nn.Linear(self.combined_dim, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, images=None, timeseries=None):
        batch_size = images.shape[0] if images is not None else timeseries.shape[0]
        # device = next(self.parameters()).device
        device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")

        # Vision branch
        if self.has_vision:
            if images is None:
                img_feat = torch.zeros(batch_size, self.vision_output_dim, device=device)
            else:
                img_feat = self.vision_model(images)
        else:
            img_feat = torch.tensor([], device=device)

        # Time-series branch
        if self.has_timeseries:
            if timeseries is None:
                ts_feat = torch.zeros(batch_size, self.ts_hidden_dim, device=device)
            else:
                if self.ts_model == 'gru':
                    _, ts_feat = self.ts_module(timeseries)
                    ts_feat = ts_feat.squeeze(0)

                elif self.ts_model == 'lstm':
                    _, (ts_feat, _) = self.ts_module(timeseries)
                    ts_feat = ts_feat.squeeze(0)

                elif self.ts_model == 'rnn':
                    _, ts_feat = self.ts_module(timeseries)  # Get last hidden state
                    ts_feat = ts_feat.squeeze(0)


                elif self.ts_model == 'cnn':
                    x = timeseries.transpose(1, 2)  # (B, C, T)
                    ts_feat = self.ts_module(x).squeeze(2)

                elif self.ts_model == 'transformer':
                    x = self.ts_proj(timeseries)                      # (B, T, 64)
                    encoded = self.ts_module(x)                      # (B, T, 64)
                    attn_weights = torch.softmax(self.ts_pool(encoded), dim=1)  # (B, T, 1)
                    ts_feat = torch.sum(attn_weights * encoded, dim=1)          # (B, 64)
        else:
            ts_feat = torch.tensor([], device=device)

        # Combine
        features = []
        if self.has_vision:
            features.append(img_feat)
        if self.has_timeseries:
            features.append(ts_feat)

        combined = torch.cat(features, dim=1)
        return self.fc(combined)

In [51]:
device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model1 = MultiModalNet(4)
model2 = MultiModalNet(4, has_timeseries=False)  # Example: only timeseries model

# Replace with your actual paths
state_dict1 = torch.load('/home/iofeidis/workspace/fedJam/fedjam-flower/runs/multimodal/multimodal_conf/multimodal_clients_10_iid/20250707-143011/final_model/multimodal_acc_09903.pth', map_location=device)
model1.load_state_dict(state_dict1)
model1.to(device)
state_dict2 = torch.load('/home/iofeidis/workspace/fedJam/fedjam-flower/runs/mobilenet_model_10_epochs/multimodal_clients_10_iid/20250709-113800/final_model/mobilenet_acc_098.pth', map_location=device)
model2.load_state_dict(state_dict2)
model2.to(device)

model1.eval()
model2.eval()


Using device: cuda:2


/home/iofeidis/miniconda3/envs/flower/lib/python3.12/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


MultiModalNet(
  (vision_model): MobileNetV3(
    (conv_stem): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNormAct2d(
      16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): Hardswish()
    )
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
          (bn1): BatchNormAct2d(
            16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): ReLU(inplace=True)
          )
          (aa): Identity()
          (se): SqueezeExcite(
            (conv_reduce): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
            (act1): ReLU(inplace=True)
            (conv_expand): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
            (gate): Hardsigmoid()
          )
          (conv_pw):

In [52]:
from flwr_datasets import FederatedDataset
from flwr_datasets.partitioner import IidPartitioner, PathologicalPartitioner

dataset_dict = None  # Needed for global reference

def load_data(partition_id: int, num_partitions: int, data_dir: str = None,
              batch_size: int = 128, classes_per_partition: int = 4,
              is_multimodal: bool = False, modality: str = "both"):
    print(f"Loading dataset {partition_id + 1} / {num_partitions}", flush=True)
    global dataset_dict

    if dataset_dict is None:
        if is_multimodal:
            dataset_dict = load_from_disk(data_dir)

            labels = sorted(set(dataset_dict["train"]["label"]))
            label2id = {lbl: i for i, lbl in enumerate(labels)}

            def encode_label(example):
                example["label"] = label2id[example["label"]]
                return example

            dataset_dict = dataset_dict.map(encode_label)
        else:
            dataset_dict = load_dataset("imagefolder", data_dir=data_dir)

    train_dataset = dataset_dict["train"]
    test_dataset = dataset_dict["test"]

    partitioner = PathologicalPartitioner(
        num_partitions=num_partitions, partition_by="label",
        num_classes_per_partition=classes_per_partition,
        class_assignment_mode="deterministic"
    )
    partitioner.dataset = test_dataset
    test_partition = partitioner.load_partition(partition_id)

    if is_multimodal:
        transform = transforms.Compose([
            CustomAugmenter(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])

        def collate_fn(batch):
            if modality in ["both", "image"]:
                images = [transform(example["image"]) for example in batch]
                images = torch.stack(images)
            else:
                images = None

            if modality in ["both", "timeseries"]:
                timeseries = [torch.tensor(example["timeseries"], dtype=torch.float32) for example in batch]
                timeseries = torch.stack(timeseries)
            else:
                timeseries = None

            labels = torch.tensor([example["label"] for example in batch])

            if modality == "both":
                return images, timeseries, labels
            elif modality == "image":
                return {"image": images, "label": labels}
            else:
                return {"timeseries": timeseries, "label": labels}

        testloader = DataLoader(test_partition, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    else:
        pytorch_transforms = transforms.Compose([
            CustomAugmenter(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])

        def apply_transforms(batch):
            batch["image"] = [pytorch_transforms(img) for img in batch["image"]]
            return batch

        test_partition = test_partition.with_transform(apply_transforms)
        testloader = DataLoader(test_partition, batch_size=batch_size)

    return testloader


In [53]:
def test_with_predictions(model, testloader, device, is_lora=False, is_timm=False,
                          is_multimodal=False):
    model.to(device)
    model.eval()
    criterion = torch.nn.CrossEntropyLoss()
    
    correct, total_loss = 0, 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in testloader:
            if is_multimodal:
                if isinstance(batch, (tuple, list)):
                    images, timeseries, labels = batch
                    images = images.to(device) if images is not None else None
                    timeseries = timeseries.to(device) if timeseries is not None else None
                    labels = labels.to(device)
                    logits = model(images, timeseries)
                else:
                    labels = batch["label"].to(device)
                    if "image" in batch:
                        logits = model(images=batch["image"].to(device))
                    else:
                        logits = model(timeseries=batch["timeseries"].to(device))
            else:
                images, labels = batch["image"], batch["label"]
                images, labels = images.to(device), labels.to(device)
                if is_timm:
                    logits = model(images)
                else:
                    logits = model(pixel_values=images).logits

            loss = criterion(logits, labels)
            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = correct / len(testloader.dataset)
    avg_loss = total_loss / len(testloader)
    return avg_loss, accuracy, all_preds, all_labels


In [54]:
test_loader = load_data(
    partition_id=0,
    num_partitions=1,
    data_dir="/home/iofeidis/workspace/hf_dataset.down75.len256.cols_all",
    batch_size=512,
    is_multimodal=True,
    modality="both"  # or "image", "timeseries"
)


Loading dataset 1 / 1


In [55]:
def test_with_predictions(model, testloader, device, is_multimodal=False):
    model.to(device)
    model.eval()
    criterion = torch.nn.CrossEntropyLoss()

    correct, total_loss = 0, 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for i, batch in enumerate(testloader):
            print(f"Processing batch {i} out of {len(testloader)}", flush=True)
            if is_multimodal:
                if isinstance(batch, (tuple, list)):
                    images, timeseries, labels = batch
                    images = images.to(device) if images is not None else None
                    timeseries = timeseries.to(device) if timeseries is not None else None
                    labels = labels.to(device)
                    logits = model(images, timeseries)
                else:
                    labels = batch["label"].to(device)
                    if "image" in batch:
                        logits = model(images=batch["image"].to(device))
                    else:
                        logits = model(timeseries=batch["timeseries"].to(device))
            else:
                images, labels = batch["image"].to(device), batch["label"].to(device)
                logits = model(images)

            loss = criterion(logits, labels)
            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = correct / len(testloader.dataset)
    avg_loss = total_loss / len(testloader)
    return avg_loss, accuracy, all_preds, all_labels


In [ ]:
loss1, acc1, preds1, labels = test_with_predictions(model1, test_loader, device, is_multimodal=True)
loss2, acc2, preds2, _      = test_with_predictions(model2, test_loader, device, is_multimodal=True)

print(f"Model 1 — Loss: {loss1:.4f}, Accuracy: {acc1:.2%}")
print(f"Model 2 — Loss: {loss2:.4f}, Accuracy: {acc2:.2%}")


Processing batch 0 out of 15


In [ ]:
target_names = ['Severe', 'Mild', 'Little/None', 'Don’t Know']  # Replace if different

print("=== Model 1 ===")
print(classification_report(labels, preds1, target_names=target_names))

print("=== Model 2 ===")
print(classification_report(labels, preds2, target_names=target_names))

fig, axs = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(labels, preds1, display_labels=target_names, ax=axs[0], cmap='Blues')
axs[0].set_title("Model 1")

ConfusionMatrixDisplay.from_predictions(labels, preds2, display_labels=target_names, ax=axs[1], cmap='Greens')
axs[1].set_title("Model 2")

plt.tight_layout()
plt.show()


In [ ]:
precision1, recall1, f1_1, _ = precision_recall_fscore_support(labels, preds1, average=None)
precision2, recall2, f1_2, _ = precision_recall_fscore_support(labels, preds2, average=None)

x = np.arange(len(target_names))
width = 0.35

plt.figure(figsize=(10, 6))
plt.bar(x - width/2, f1_1, width, label='Model 1', color='skyblue')
plt.bar(x + width/2, f1_2, width, label='Model 2', color='lightgreen')
plt.xticks(x, target_names)
plt.ylabel('F1 Score')
plt.title('Per-Class F1 Score Comparison')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
